# 📊 Publication-Grade Exploratory Data Analysis (EDA)
## CodeAlpha Internship — Task 1: Credit Scoring Model

### 1. Library Imports & Visual Theme Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
sns.set_theme(style="whitegrid", palette="muted")
pd.set_option('display.max_columns', None)

### 2. Dataset Ingestion

In [ ]:
train_df = pd.read_csv('train.csv', low_memory=False)
test_df = pd.read_csv('test.csv', low_memory=False)

In [ ]:
print(f"Train Data Shape: {train_df.shape}")
print(f"Test Data Shape:  {test_df.shape}")

### 3. Data Structure Inspection

In [ ]:
train_df.head(3)

In [ ]:
train_df.info()

### 4. Target Label Distribution Profiling

In [ ]:
target_counts = train_df['Credit_Score'].value_counts()
target_percentages = train_df['Credit_Score'].value_counts(normalize=True) * 100

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=target_counts.index, y=target_counts.values, ax=ax, palette=['#3498db', '#e74c3c', '#2ecc71'])
ax.set_title('Target Class Absolute Frequency (Credit_Score)', fontsize=12, fontweight='bold')
ax.set_ylabel('Record Count')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', fontsize=11, color='white', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(target_percentages, labels=target_percentages.index, autopct='%1.1f%%', startangle=140,
       colors=['#3498db', '#e74c3c', '#2ecc71'], explode=(0.03, 0.03, 0.03),
       textprops={'fontsize': 11, 'fontweight': 'bold'})
ax.set_title('Target Class Relative Share (%)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 5. Missingness Audit

In [ ]:
missing_counts = train_df.isnull().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)

In [ ]:
missing_percentages = (missing_counts / len(train_df)) * 100
missing_summary_df = pd.DataFrame({'Missing Count': missing_counts, 'Percentage (%)': missing_percentages})

In [ ]:
plt.figure(figsize=(12, 6))
ax = sns.barplot(x=missing_summary_df['Percentage (%)'], y=missing_summary_df.index, palette='viridis')
plt.title('Missing Value Percentage by Feature', fontsize=14, fontweight='bold')
plt.xlabel('Percentage Missing (%)')
plt.ylabel('Features')
for p in ax.patches:
    width = p.get_width()
    ax.annotate(f'{width:.2f}%', (width + 0.3, p.get_y() + p.get_height() / 2),
                ha='left', va='center', fontsize=10)
plt.xlim(0, max(missing_summary_df['Percentage (%)']) + 3)
plt.tight_layout()
plt.show()

### 6. String Sanitization & Regex Utilities

In [ ]:
def sanitize_numeric(val):
    if pd.isna(val):
        return np.nan
    clean_str = str(val).strip().replace('_', '')
    try:
        parsed = float(clean_str)
        return parsed if parsed >= 0 else np.nan
    except:
        return np.nan

In [ ]:
def parse_credit_history_months(val):
    if pd.isna(val) or val == '_':
        return np.nan
    match = re.search(r'(\d+)\s*Years\s*and\s*(\d+)\s*Months', str(val))
    if match:
        return int(match.group(1)) * 12 + int(match.group(2))
    return np.nan

### 7. Feature Parsing & Cleaning Execution

In [ ]:
numeric_corrupted_cols = [
    'Age', 'Annual_Income', 'Num_of_Loan', 'Num_of_Delayed_Payment',
    'Changed_Credit_Limit', 'Outstanding_Debt', 'Amount_invested_monthly', 'Monthly_Balance'
]

In [ ]:
for df in [train_df, test_df]:
    for col in numeric_corrupted_cols:
        df[col] = df[col].apply(sanitize_numeric)
    df['Credit_History_Age_Months'] = df['Credit_History_Age'].apply(parse_credit_history_months)
    df['Occupation'] = df['Occupation'].replace('_______', np.nan)
    df['Credit_Mix'] = df['Credit_Mix'].replace('_', np.nan)
    df['Payment_of_Min_Amount'] = df['Payment_of_Min_Amount'].replace('NM', np.nan)

### 8. Numerical Distribution & Outlier Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=train_df, x='Credit_Score', y='Annual_Income', ax=ax, palette='Blues_r', showfliers=False)
ax.set_title('Annual Income Distribution by Credit Score', fontsize=12, fontweight='bold')
ax.set_ylabel('Annual Income ($)')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=train_df, x='Credit_Score', y='Outstanding_Debt', ax=ax, palette='Reds_r', showfliers=False)
ax.set_title('Outstanding Debt Distribution by Credit Score', fontsize=12, fontweight='bold')
ax.set_ylabel('Outstanding Debt ($)')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=train_df, x='Credit_Score', y='Credit_History_Age_Months', ax=ax, palette='Greens_r', showfliers=False)
ax.set_title('Credit History Length (Months) by Credit Score', fontsize=12, fontweight='bold')
ax.set_ylabel('Credit History (Months)')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=train_df, x='Credit_Score', y='Interest_Rate', ax=ax, palette='Purples_r', showfliers=False)
ax.set_title('Interest Rate Charged by Credit Score', fontsize=12, fontweight='bold')
ax.set_ylabel('Interest Rate (%)')
plt.tight_layout()
plt.show()

### 9. Categorical Attributes Profiling

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.countplot(data=train_df, x='Credit_Mix', hue='Credit_Score', ax=ax, palette='magma')
ax.set_title('Credit Mix Quality vs. Credit Score', fontsize=12, fontweight='bold')
ax.set_xlabel('Credit Mix')
ax.set_ylabel('Record Count')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.countplot(data=train_df, y='Payment_Behaviour', hue='Credit_Score', ax=ax, palette='crest')
ax.set_title('Payment Behavior Category vs. Credit Score', fontsize=12, fontweight='bold')
ax.set_xlabel('Record Count')
ax.set_ylabel('Payment Behavior Category')
plt.tight_layout()
plt.show()

### 10. High-Resolution Correlation Heatmap

In [ ]:
numeric_features = train_df.select_dtypes(include=[np.number]).columns
corr_matrix = train_df[numeric_features].corr()

In [ ]:
plt.figure(figsize=(15, 11))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1,
            linewidths=0.5, cbar_kws={'label': 'Pearson Correlation Coefficient'})
plt.title('High-Resolution Feature Correlation Matrix Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()